Utils

In [ ]:
!pip uninstall torchcrf -y
!pip install pytorch-crf
!pip install -q transformers datasets accelerate seqeval torch
!pip install -q transformers datasets accelerate seqeval torch
!pip install -q git+https://github.com/csebuetnlp/normalizer.git

Encoder Baseline for training BERT models on Clean Training Set

In [ ]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback)
from seqeval.metrics import f1_score, precision_score, recall_score
from normalizer import normalize # For BanglaBERT normalization
from pathlib import Path


# CONFIGURATION
MODEL_NAME = "xlm-roberta-base" # Alternatively => "csebuetnlp/banglabert"
USE_CRF = True   # True = Encoder + CRF, False = Encoder + Softmax

# clean data (annoatated) for training
DATA_DIR = Path("data/splits/clean")
TRAIN_FILE = DATA_DIR / "clean_train.jsonl"
VAL_FILE = DATA_DIR / "clean_val.jsonl"


OUTPUT_DIR = Path("outputs") / f"{MODEL_NAME.split('/')[-1]}_{'crf' if USE_CRF else 'softmax'}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "validation": VAL_FILE})
train_ds = dataset["train"]
val_ds = dataset["validation"]
print(f"Train size: {len(train_ds)} sentences; Val size: {len(val_ds)} sentences")


label_set = {"O"}
for ex in train_ds:
    label_set.update(ex["labels"])

new_i = {l.replace("B-", "I-") for l in label_set if l.startswith("B-")}
label_set.update(new_i)

label_list = ["O"] + sorted([l for l in label_set if l != "O"])
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

label_info = {
    "label_list": label_list,
    "label2id": label2id,
    "id2label": id2label
}


In [ ]:

# TOKENIZATION

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    normalized_tokens = [[normalize(tok) for tok in seq] for seq in examples["tokens"]]

    tokenized = tokenizer(
        normalized_tokens,
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    aligned_labels = []
    for i, labels in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100) # Special tokens ignored
            elif word_id != prev_word_id:
                # Fallback to "O" if a rare label somehow appears only in validation
                label_ids.append(label2id.get(labels[word_id], label2id["O"]))
            else:
                label = labels[word_id]
                if label.startswith("B-"):
                    label = label.replace("B-", "I-")
                label_ids.append(label2id.get(label, label2id["O"]))
            prev_word_id = word_id
        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized

train_tok = train_ds.map(tokenize_function, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_function, batched=True, remove_columns=val_ds.column_names)



class EncoderTokenClassifier(nn.Module):
    def __init__(self, model_name, num_labels, use_crf=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.encoder.config.hidden_size, num_labels)
        self.use_crf = use_crf

        if use_crf:
            self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        emissions = self.classifier(sequence_output)
        mask = attention_mask.bool()

        # --- CRF MODE ---
        if self.use_crf:
            if labels is not None:
                valid_labels = labels.clone()
                valid_labels[valid_labels == -100] = 0 # CRF cannot handle -100
                loss = -self.crf(emissions, valid_labels, mask=mask, reduction="mean")
                return {"loss": loss, "logits": emissions}
            else:
                prediction = self.crf.decode(emissions, mask=mask)
                max_len = emissions.size(1)
                padded = [seq + [-100] * (max_len - len(seq)) for seq in prediction]
                return {"logits": torch.tensor(padded, device=emissions.device)}

        # --- SOFTMAX MODE ---
        else:
            logits = emissions
            if labels is not None:
                loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
                loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))
                return {"loss": loss, "logits": logits}
            else:
                # 🚨 FIX 4: Return raw logits, NOT argmax. Let metrics handle it.
                return {"logits": logits}


def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    if isinstance(predictions, tuple):
        predictions = predictions[0]
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    true_preds, true_labels = [], []

    for pred_seq, label_seq in zip(predictions, labels):
        cp, cl = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                cp.append(id2label[int(p)])
                cl.append(id2label[int(l)])
        true_preds.append(cp)
        true_labels.append(cl)

    return {
        "f1": f1_score(true_labels, true_preds),
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "macro_f1": f1_score(true_labels, true_preds, average="macro"), # macro F1 for early stopping and the main metric to track
    }

class CRFTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs, labels=labels)
        return (outputs["loss"], outputs) if return_outputs else outputs["loss"]

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad():
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            logits = outputs["logits"]
        return (None, logits, labels)

    def save_model(self, output_dir=None, _internal_call=False):
        output_dir = output_dir or self.args.output_dir
        os.makedirs(output_dir, exist_ok=True)
        torch.save(self.model.state_dict(), f"{output_dir}/pytorch_model.bin")
        self.model.encoder.config.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)

        with open(f"{output_dir}/label_info.json", "w") as f:
            json.dump(label_info, f)

        # Save custom config
        custom_config = {
            "model_name": MODEL_NAME,
            "use_crf": self.model.use_crf,
            "num_labels": len(label_list)
        }
        with open(f"{output_dir}/custom_config.json", "w") as f:
            json.dump(custom_config, f)


model = EncoderTokenClassifier(model_name=MODEL_NAME, num_labels=len(label_list), use_crf=USE_CRF)



In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=15,         
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1", # Important: Use macro F1 for early stopping and best model selection
    fp16=torch.cuda.is_available(),
    report_to="none",
    optim="adamw_torch",
    warmup_ratio=0.1,
)

trainer = CRFTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

mode_str = "CRF" if USE_CRF else "Softmax"
print(f"Training {MODEL_NAME} with {mode_str} on Clean Dataset")
trainer.train()

# Final Save
final_dir = f"{OUTPUT_DIR}/final"
trainer.save_model(final_dir)
print(f"Baseline model saved to {final_dir}")